# Logistic Regression - Schrifterkennung

2026-06-16 ug Version 1.4

Mit Hilfe der **Logistischen Regression** wollen wir die Ziffern 0 bis 9 in kleinen Bildern erkennen.

- Eingang `X` (feature): Bild mit 8x8 Pixeln  (64 einzelne Pixel mit jeweils einem Intensitätswert)
- Ausgang `y` (target): 10 Klassen für die Ziffern für 0 bis 9

Dies ist ein Beispiel für eine Mehrklassenklassifizierung (multiclass classification).

-----
Verwendeter Datensatz: [sklearn-digits-Datensatz](
https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)
<br>
Der Datensatz ist im scikit-learn Python Package enthalten.
<br> 
Der Datenbesteht aus 1797 einzelnen Bildern, die als Eingänge (features) für unsere Modell der Logistischen Regression dienen.
Jedes Bild besteht aus 64 Bildpunkten Pixeln, die als 8x8 Matrix angeordnet sind.
Jeder Bildpunkt Pixel enthält einen Helligkeitswert (Intensität).
Die Bilder stellen Ziffern von 0 bis 9 dar; also 10 Klassen.
Welche Klasse in welchem Bild dargestellt wird, gibt der Ausgang (target oder label) an.
Der Datensatz enthält ungefähr gleich viele Bilder von den einzelnen Ziffern.
  

-----
Referenzen:
- [RealPython- logistic regression python](https://realpython.com/logistic-regression-python)
- [Codebasics - Machine Learning Tutorial Python - 8 Logistic Regression (Multiclass Classification)](<https://www.youtube.com/watch?v=J5bXOOmkopc&list=PLeo1K3hjS3uvCeTYTeyfe0-rN5r8zn9rw>)
- [sklearn - Recognizing hand-written digits](https://scikit-learn.org/stable/auto_examples/classification/plot_digits_classification.html#sphx-glr-auto-examples-classification-plot-digits-classification-py)


<!--
### Plotten der Confusion Matrix

Umgestellt von `plot_confusion_matrix()` auf `sklearn.metrics.ConfusionMatrixDisplay`

[`plot_confusion_matrix()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.plot_confusion_matrix.html)

Function plot_confusion_matrix is deprecated; Function `plot_confusion_matrix` is deprecated in 1.0 and will be removed in 1.2. Use one of the class methods: ConfusionMatrixDisplay.from_predictions or ConfusionMatrixDisplay.from_estimator.

```
fig, ax = plt.subplots(figsize=(10, 7))
disp = plot_confusion_matrix(model, x_test, y_test,ax=ax)
disp.figure_.suptitle("Confusion Matrix")
print(f"Confusion matrix:\n{disp.confusion_matrix}")
```

-->

---
## Step 1: Importieren der Python Pakete

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import pandas as pd

---
## Step 2a: Daten laden

Datensatz: [sklearn-digits-Datensatz](
https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)

Each datapoint is a 8x8 image of a digit.


    Classes              10
    Samples per class  ~180
    Samples total      1797
    Dimensionality       64


In [ ]:
X, y = load_digits(return_X_y=True)

### Überblick über die Daten verschaffen
Eingang `X`:

In [ ]:
type(X)

Der Eingang `X` ist vom Typ `numpy.ndarray` und hat die Gestalt:

In [ ]:
X.shape

1797 Bilder mit jeweils 64 Werten;
<br>
ein Bild besteht aus 64 Pixeln; 64 Pixel werden als 8x8 Pixeln zu einem Bild angeordnet; jeder Pixel hat einen Intensitätswert.

In [ ]:
X.dtype

Wertebereich der Daten von `X`

In [ ]:
X.min(),X.max()

Die Pixel haben ein Intesitätswert der vom Datentyp `float64` ist und einen Wertebereich von 0 bis 16 hat.
<br>
Ausgang `y`:

In [ ]:
type(y)

Der Ausgang `y` ist auch vom Typ `numpy.array` und hat die Gestalt:

In [ ]:
y.shape

In [ ]:
y.min(),y.max()

10 Klassen für die Ziffern für 0 bis 9

#### Wie häufig ist welche Klasse im Datensatz vertreten?

Wir wandeln das Numpy Array `y` in ein Pandas Series-Objekt um und wenden dann die [`.value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)-Methode an, um die absolute Häufigkeit der 10 Klassen (Ziffern von 0 bis 9) zu erhalten

In [ ]:
df=pd.Series(y)
df.value_counts()

Die Ziffer 3 kommt mit 183 Bildern am häufigsten vor; die Ziffer 8 kommt mit 174 am wenig häufigsten vor. Dies ist eine Differenz von 9 Bildern. Die Ziffern sind im Datensatz annährend gleichmäßig verteilt.

#### erstes Bild darstellen
Wir wollen uns das erste Bild `X[0]` darstellen:

In [ ]:
X[0]

Mit der Methode [`.reshape()`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) bauen wir den Vektor mit 64 Werten in eine Matrix mit 8 Zeilen und 8 Spalten um:

In [ ]:
X[0].reshape(8,8)

Mit der Matplotlib-Funktion [`plt.imshow()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)
kann eine Matrix als Bild dargestellt werden:


Mit der Matplotlib-Funktion [`plt.gray()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.gray.html)
kann die Farbtabelle auf Graustufen umgestellt werden.

In [ ]:
plt.gray()
plt.imshow(X[0].reshape(8,8))

die Klasse des ersten Bildes 

In [ ]:
y[0]

#### Darstellen der ersten 10 Bilder

Mit der Methode [`.ravel()`](https://numpy.org/doc/stable/reference/generated/numpy.ravel.html)
wird eine mehrdimensionales Array in ein ein-dimesionales Array gewandelt.



to ravel - sich verwickeln 

In [ ]:
fig, axs = plt.subplots(ncols=5,nrows=2,figsize=(16,8))
print(axs.shape)
for k,ax in enumerate(axs.ravel()):
    ax.imshow(X[k].reshape(8,8))
    ax.axis('off')                         # Achsen unterdrücken
    ax.set_title(f'y={y[k]}',fontsize=18)
    
fig.tight_layout()

----
### Step 2b: Aufteilen der Daten in Trainings- und Validierungsdatensatz

Mit Sklearn-Funktion [`train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
kann ein Datensatz in einen Trainings- und einen Test/Validierungsdatensatz aufgeteilt werden.

Mit dem Parameter `test_size` kann die Anteil zwischen 0 und 1 der Test-Daten festleget werden.

Die Verteilung erfolgt durch einen Zufallsgenerator.
Um den reproduzierbares Verhalten in diesem Notebook zu bekommen, wird der Zufallsgenerator mit einem Wert initialisiert (`random_state=0`).




In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
y_train.shape, y_test.shape

In [ ]:
X_train.min(), X_train.max()

### Step 2c: Skalierung der Daten

Bei der Skalierung wird von den einzelnen Eingangsvariablen ihr jeweiliger Mittelwert abgezogen und anschließend wird durch die Standardabweichung geteilt.
Dadurch werden die Eingangsvariablen mittelwertfrei mit einer Varianz von eins.
$$
    z = \frac{x - u}{s}
$$
mit $u$ dem Mittelwert und $s$ der Standardabweichung.

Durch die Skalierung werden die Daten gleichmäßiger gemacht, was sich vorteilhaft auf die numerischen Berechungen auswirkt.

Mit [`StandardScaler()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
wird zunächst ein Objekt `scaler` angelegt und mit dessen `.fit_transform()`-Methode werden die Trainingsdaten dann wie oben beschrieben bearbeitet.

Das Objekt `scaler` wird nachher noch für die Bearbeitung des Validierungsdatensatzes benötigt.


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

In [ ]:
X_train.min(), X_train.max()

----
### Step 3: Modell angelegen und Modell an die Daten anpassen


Current solver='liblinear' now rejects n_classes >= 3 in this sklearn version.

Fix:

Use solver='lbfgs' (works for multinomial multiclass).

 3) Keep prior hyperparameters where valid:
    - Keep C=1, random_state=0.
 4) Improve convergence stability:
    - Set max_iter high enough (e.g., 2000).


Parameter (Hyperparameter) von [`LogisticRegression()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html):
- `solver=` - Optimierungsalgorithmus 
- `C=`   - Regularisierung - kleinere Werte bewirken eine stärkere Regularierung ("Modellparamter klein halten")

veraltet:
- `multi_class='ovr'` - die Anpassung wird für jede Klasse separat durchgeführt
    label.
    


In [ ]:
#model = LogisticRegression(solver='liblinear', C=1,
#                           random_state=0)

model = LogisticRegression(
    solver='lbfgs',
    C=1,
    random_state=0,
    max_iter=2000
)

model.fit(X_train, y_train)

Klassen des Modells anzeigen:

In [ ]:
model.classes_

---
### Step 4: Model beurteilen

Die Eingangsvariablen aus dem Testdatensatz müssen auch skaliert werden. 

In [ ]:
X_test = scaler.transform(X_test)

Das Modell wird genutzt um die Ausgänge des Testdatensatzes zu prädizieren:

In [ ]:
y_pred = model.predict(X_test)

Das Modell kann sehr gut den Trainingsdatensatz voraussagen:


In [ ]:
model.score(X_train, y_train)

Wir schauen jetzt wie gut das Modell die Ausgänge des Testdatensatz voraussagen kann:

In [ ]:
model.score(X_test, y_test)

#### Confusion Matrix

Berechnen der [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

plotten der Confusion Matrix mit [`seaborn.heatmap()`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
plt.figure(figsize = (10,7))
sns.heatmap(cm, annot=True, cmap='coolwarm')
plt.xlabel('Predicted outputs')
plt.ylabel('Actual outputs')

#### Plotten der Confusion Matrix mit `ConfusionMatrixDisplay`

Mit [`sklearn.metrics.ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) wird im folgenden auf drei verschienden Weise, die Confusion Matrix als ein Heatmap-Plot dargestellt:
- Möglichkeit 1: confusion matrix liegt vor
- Möglichkeit 2: Modell (Schätzer/Estimator) liegt vor; Datensatz (Test-Datensatz) wird für die Berechnung benötigt
- Möglichkeit 3: Prädizierte Ausgangswerte liegen vor; es werden noch die wahren Ausgangswerte dazu benötigt

_**Möglichkeit 1**_: confusion matrix liegt vor

Instanz der [`sklearn.metrics.ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html)-Klasse erstellen und mit der [.plot()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.plot)-Methode den Plot der Confusion Matrix erzeugen.

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot()

_**Möglichkeit 2**_: Modell (Schätzer/Estimator) liegt vor; Datensatz (Test-Datensatz) wird für die Berechnung benötigt

[`.from_estimator()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_estimator)-Methode:
- 1. Parameter: Modell welches an die Traingsdaten angepasst wurde
- 2. und 3. Parameter: Daten für die die Conflusion Matrix berechnet werden soll 

In [ ]:
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)

_**Möglichkeit 3**_: Prädizierte Ausgangswerte liegen vor; es werden noch die wahren Ausgangswerte dazu benötigt

[`.from_predictions()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)-Methode:
- 1. Parameter wahre Ausgangswerte (true targets)
- 2. Parameter prädizierte Ausgangswerte (predicted targts)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

#### Classification report

In [ ]:
print(classification_report(y_test, y_pred))